In [ ]:
# run this notebook after 1_methods_relatedness_python
# use this notebook to identify parent-offspring, siblings, trios, twins, and quads 
# after this, run 3_call_ibd2_using_snipar_python

In [ ]:
library(data.table)
library(dplyr)
library(tidyverse)
library(bigrquery)

In [ ]:
king = fread("./relatedness_dec20/king_sib_par_dec20.kin0", sep = "\t", header = TRUE)
king_rev = king
king_rev$ID1 = king$ID2
king_rev$ID2 = king$ID1
king = rbind(king, king_rev)
king = king %>% filter(ID1 < ID2)

In [ ]:
relatedness = fread("./king_files/relatedness.tsv", sep = "\t", header = TRUE)
relatedness_rev = relatedness
relatedness_rev[,1] = relatedness[,2]
relatedness_rev[,2] = relatedness[,1]
relatedness = rbind(relatedness, relatedness_rev)
relatedness = relatedness %>% filter(i.s < j.s)

In [ ]:
flagged = fread("./flagged_samples.tsv", sep = "\t", header=TRUE)

In [ ]:
# remove flagged individuals and generate list of siblings 

king = king %>% filter(!(king$ID1 %in% flagged$s) & !(king$ID2 %in% flagged$s))
relatedness = relatedness %>% filter(!(relatedness$i.s %in% flagged$s) & !(relatedness$j.s %in% flagged$s))
sibs = king %>% filter(InfType == "FS") %>% select(ID1, ID2) %>% unique()
sibs$pair = paste(sibs$ID1, sibs$ID2, sep ="_"
        )

In [ ]:
pars = king %>% filter(InfType == "PO") %>% select(ID1, ID2) %>% unique()
twin_dup = relatedness %>% filter(kin > 0.354)
names(twin_dup) = c("IID1", "IID2", "kin")
# I'll only keep the ones in IID1 in the twin_dup file 
fwrite(twin_dup, "./relatedness_dec20/twin_dup.txt",sep = "\t", quote = FALSE, row.names = FALSE)

In [ ]:
# Find connected components (families of twins/dups)
library(igraph)
g = graph_from_data_frame(twin_dup[, c("IID1", "IID2")], directed = FALSE)
comp = components(g)

# For each family, keep one random individual, remove the rest
set.seed(42)
twin_dup_remove = c()
for (fam in seq_len(comp$no)) {
  members = names(comp$membership[comp$membership == fam])
  keep = sample(members, 1)
  twin_dup_remove = c(twin_dup_remove, setdiff(members, keep))
}

In [ ]:
# remove twin / dup from sib file 
sibs$remove = FALSE
for (i in 1:nrow(sibs)){
    sibs$remove[i] = sibs$ID1[i] %in% twin_dup_remove | sibs$ID2[i] %in% twin_dup_remove
}
sum(sibs$remove)
sibs = sibs %>% filter(!remove)
fwrite(sibs, "./relatedness_dec20/sibs.txt", quote = F, row.names = F, sep = "\t", col.names = T)

In [ ]:
# remove twin / dup from par file 
pars$remove = FALSE
for (i in 1:nrow(pars)){
    pars$remove[i] = pars$ID1[i] %in% twin_dup_remove | pars$ID2[i] %in% twin_dup_remove
}
sum(pars$remove)
pars = pars %>% filter(!remove)

In [ ]:
# add age and sex to pars file 
age_sex = fread("./king_files/age_sex.txt", sep = ",", header = TRUE)
# clean age_sex dob 
age_sex$dob = as.numeric(age_sex$date_of_birth <- sub("-.*", "", age_sex$date_of_birth))
# annotate 
pars$IID1_dob = NA
pars$IID2_dob = NA
pars$parent = NA
pars$offspring = NA
for (i in 1:nrow(pars)){
    pars$IID1_dob[i] = age_sex$dob[which(age_sex$person_id == pars$ID1[i])]
    pars$IID2_dob[i] = age_sex$dob[which(age_sex$person_id == pars$ID2[i])]
    pars$parent[i] = ifelse(pars$IID1_dob[i] < pars$IID2_dob[i], pars$ID1[i], pars$ID2[i])
    pars$offspring[i] = ifelse(pars$IID1_dob[i] > pars$IID2_dob[i], pars$ID1[i], pars$ID2[i])
}
fwrite(pars, "./relatedness_dec20/pars.txt", quote = F, row.names = F, sep = "\t")
pars = pars %>% filter(abs(IID1_dob-IID2_dob) > 10)
fwrite(pars, "./relatedness_dec20/pars.txt", quote = F, row.names = F, sep = "\t")

In [ ]:
# add dob to sib file 
sibs$IID1_dob = NA
sibs$IID2_dob = NA
for (i in 1:nrow(sibs)){
    sibs$IID1_dob[i] = age_sex$dob[which(age_sex$person_id == sibs$ID1[i])]
    sibs$IID2_dob[i] = age_sex$dob[which(age_sex$person_id == sibs$ID2[i])]
}
sibs = sibs %>% filter(abs(sibs$IID1_dob-sibs$IID2_dob) < 30)
nrow(sibs)
fwrite(sibs, "./relatedness_dec20/sibs.txt", quote = F, row.names = F, sep = "\t", col.names = T)

In [ ]:
# identify families 
library(igraph)

# 1. Collect all offspring IDs
all_offspring= unique(c(sibs$ID1, sibs$ID2, pars$offspring))
pars$offspring = as.character(pars$offspring)
pars$parent = as.character(pars$parent)
sibs$ID1 = as.character(sibs$ID1)
sibs$ID2 = as.character(sibs$ID2)

# Build connected components from sibs ONLY
all_offspring = unique(c(sibs$ID1, sibs$ID2, pars$offspring))

g = graph_from_data_frame(sibs[, c("ID1", "ID2")], directed = FALSE,
                           vertices = data.frame(name = all_offspring))
comp = components(g)

offspring_map = data.frame(id = names(comp$membership),
                            family = unname(comp$membership),
                            stringsAsFactors = FALSE)

# Attach parents
parent_map = pars %>%
  inner_join(offspring_map, by = c("offspring" = "id")) %>%
  distinct(id = parent, family)

# Build family list
families = list()
for (fam in sort(unique(offspring_map$family))) {
  off = offspring_map$id[offspring_map$family == fam]
  par = parent_map$id[parent_map$family == fam]
  families[[fam]] <- list(parents = par, offspring = off)
}

In [ ]:
# Remove families with just 1 parent and 1 offspring
families = Filter(function(fam) {
  !(length(fam$parents) == 1 && length(fam$offspring) == 1)
}, families)

In [ ]:
# Re-assign family IDs
for (i in seq_along(families)) {
  families[[i]]$fid = paste0("F", i)
}

# if a parent i s in multiple families, remove one of those randomly 
# Collect all parents across families
parent_fam = do.call(rbind, lapply(families, function(fam) {
  if (length(fam$parents) > 0) {
    data.frame(parent = fam$parents, fid = fam$fid, stringsAsFactors = FALSE)
  }
}))

set.seed(42)
repeat {
  parent_fam = do.call(rbind, lapply(families, function(fam) {
    if (length(fam$parents) > 0) {
      data.frame(parent = fam$parents, fid = fam$fid, stringsAsFactors = FALSE)
    }
  }))
  
  dupe_parents = parent_fam %>% group_by(parent) %>% filter(n() > 1) %>% ungroup()
  if (nrow(dupe_parents) == 0) break
  
  fids_to_keep = dupe_parents %>%
    group_by(parent) %>%
    slice_sample(n = 1) %>%
    ungroup() %>%
    pull(fid)
  
  fids_to_remove = dupe_parents %>%
    filter(!fid %in% fids_to_keep) %>%
    pull(fid) %>%
    unique()
  
  families = Filter(function(fam) !(fam$fid %in% fids_to_remove), families)
}

for (i in seq_along(families)) {
  families[[i]]$fid <- paste0("F", i)
}

In [ ]:
# Build trios dataframe: families with exactly 2 parents and ≥1 offspring
trios <- do.call(rbind, lapply(families, function(fam) {
  if (length(fam$parents) == 2 && length(fam$offspring) >= 1) {
    data.frame(
      fid = fam$fid,
      par1 = fam$parents[1],
      par2 = fam$parents[2],
      offspring = fam$offspring,
      stringsAsFactors = FALSE
    )
  }
}))

In [ ]:
# identify quads 
sibs <- do.call(rbind, lapply(families, function(fam) {
  if (length(fam$offspring) >= 2) {
    pairs <- combn(sort(fam$offspring), 2)
    data.frame(fid = fam$fid,
               idv1 = pairs[1, ],
               idv2 = pairs[2, ],
               par1 = fam$parents[1],
               par2 = fam$parents[2],
               stringsAsFactors = FALSE)
  }
}))

quads <- do.call(rbind, lapply(families, function(fam) {
  if (length(fam$parents) == 2 && length(fam$offspring) >= 2) {
    pairs <- combn(sort(fam$offspring), 2)
    data.frame(par1 = fam$parents[1],
               par2 = fam$parents[2],
               off1 = pairs[1, ],
               off2 = pairs[2, ],
               fid = fam$fid,
               stringsAsFactors = FALSE)
  }
}))

In [ ]:
fwrite(quads, "./relatedness_dec20/quads_new.txt", quote = F, row.names = F, sep = "\t")
fwrite(trios, "./relatedness_dec20/trios_final.txt", quote = F, row.names = F, sep = "\t")
fwrite(sibs, "./relatedness_dec20/sibs_final.txt", quote = F, row.names = F, sep = "\t")

In [ ]:
# make age sex file for snipar 
age_sex_snipar = data.frame(FID = 0, IID = unique(c(king_snipar$ID1, king_snipar$ID2)), sex = NA, age = NA)
for (i in 1:nrow(age_sex_snipar)){
    sub = age_sex %>% filter(person_id == age_sex_snipar$IID[i])
    age_sex_snipar$sex[i] = sub$sex_at_birth[1]
    age_sex_snipar$age[i] = 2025 - sub$dob[1]
}
age_sex_snipar$sex = ifelse(age_sex_snipar$sex == 'Female', 'F', 'M')
old = age_sex_snipar
old$FID = old$IID
old$keep = NA
for (i in 1:nrow(old)){
    old$keep[i] = agesex$sex_at_birth[which(agesex$person_id == old$IID[i])] %in% c("Female", "Male")
}
old = old %>% filter(keep)
fwrite(old, "./sibs_agesex.txt", sep = "\t", row.names=FALSE, quote=FALSE)